### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="video_transcoding_time_prediction",
    dataset_year="2015",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C58C9K",
    download_description="""
wget https://archive.ics.uci.edu/static/public/335/online+video+characteristics+and+transcoding+time+dataset.zip \
&& unzip online+video+characteristics+and+transcoding+time+dataset.zip transcoding_mesurment.tsv \
&& rm online+video+characteristics+and+transcoding+time+dataset.zip \
&& mkdir -p local-data-warehouse/video_transcoding_time_prediction \
&& mv transcoding_mesurment.tsv local-data-warehouse/video_transcoding_time_prediction/
""",
    # References
    academic_reference_bibtex="""@inproceedings{deneke2014video,
  title={Video transcoding time prediction for proactive load balancing},
  author={Deneke, Tewodors and Haile, Habtegebreil and Lafond, S{\'e}bastien and Lilius, Johan},
  booktitle={2014 IEEE International Conference on Multimedia and Expo (ICME)},
  pages={1--6},
  year={2014},
  organization={IEEE}
}
""",
    academic_reference_bibtex_key="deneke2014video",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the tabular prediction task from the UCI archive, which is a video transcoding time prediction task. The data is non-IID and grouped by video, with multiple transcoding measurements per video. The target variable is the transcoding time, and the features include various characteristics of the videos and transcoding settings.

- Some features on the videos (like URL and category) are not part of this dataset as we got it from UCI, but this sounds like a good idea for the task.
- The data includes several transcoding per video (also with different output target formats), which creates a non-IID setting. We will use the video id as the group label, and we will make sure to split the data such that all transcoding measurements for a given video are in the same split (train/test).
- We log scale the target variable.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_transcoding_time",
    problem_type="regression",
    objective_metric_name="rmse",
    # For grouped data
    group_on="video_id",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "transcoding_mesurment.tsv", sep="\t")
print("Loaded data shape:", df.shape)

df = df.drop(columns=[
    # Target colum / leaking
    "umem",
    # Constant
    "b_size",
])
df = df.rename(columns={
    "utime": "log_transcoding_time",
    "id": "video_id",
})

as_cat_dtype = [
    "video_id",
    "codec",
    "o_codec",
]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")
df["log_transcoding_time"] = np.log(df["log_transcoding_time"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (68784, 22)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 68,784
Columns: 20
Use sampling: False (sample size: 68,784)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['video_id', 'size', 'i_size', 'p_size', 'bitrate', 'duration', 'frames', 'p', 'i', 'framerate']
Rows remaining as candidates after top-10 filter: 67,766 (of 68,784)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 81 (0.12% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


/home/lennart_priorlabs_ai/.venvs/tabarena_1503/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
# Sample Rows
df_head

,video_id,duration,codec,width,height,bitrate,framerate,i,p,b,frames,i_size,p_size,size,o_codec,o_bitrate,o_framerate,o_width,o_height,log_transcoding_time
0,4b7b-FY4JPI,203.198,flv,320,240,239336,29.000000,104,5974,0,6078,853613,5225468,6079081,flv,820000,25.00,1280,720,1.003569
1,2SkA-cUEaJ0,490.600,h264,480,360,300437,25.000000,256,12010,0,12266,1448177,16976151,18424328,vp8,3000000,25.00,1920,1080,3.832265
2,2TXm-QHXCzU,270.655,vp8,640,480,652967,30.062963,89,8028,0,8117,1912240,20178890,22091130,flv,3000000,15.00,480,360,0.740031
3,1ZzJ-T2FJAw,250.017,h264,320,240,153059,29.000000,128,7366,0,7494,856855,3926589,4783444,flv,242000,29.97,176,144,-0.345311
4,1WrA-SRsgEk,384.451,h264,640,480,646432,29.000000,205,11318,0,11523,2839955,28225248,31065203,flv,539000,12.00,176,144,0.408128


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,video_id,category,0.0,0.0,1099.0,"3o_2-igmsyo, 3mWp-QQn-1M, 3fpg-LlHcks, 3fmy-4uvR9U, 3dI-kJGuWbY, 3_fF9rtYCQ0, 3YNh-PI2q64, 3Xyf-6FdtDA, 3Wdg-dsGihA, 3U8m-ETsTS4"
1,codec,category,0.0,0.0,4.0,"h264, vp8, mpeg4, flv"
2,o_codec,category,0.0,0.0,4.0,"mpeg4, vp8, flv, h264"
3,duration,float64,0.0,0.0,1086.0,"395.44, 106.765, 256.2083, 176.901, 476.957, 69.9383, 130.3567, 368.32, 33.09, 39.055"
4,framerate,float64,0.0,0.0,261.0,"29.0, 12.0, 25.0, 30.0, 15.0, 23.0, 24.0, 7.0, 13.0, 16.0"
5,o_framerate,float64,0.0,0.0,5.0,"15.0, 12.0, 29.97, 25.0, 24.0"
6,log_transcoding_time,float64,0.0,0.0,10960.0,"-0.0367, 0.2183, 0.2151, 0.2531, 0.2311, -0.0325, -0.0121, -0.3975, 0.2086, -0.0408"
7,width,int64,0.0,0.0,6.0,"480, 320, 176, 1280, 640, 1920"
8,height,int64,0.0,0.0,6.0,"360, 240, 144, 720, 480, 1080"
9,bitrate,int64,0.0,0.0,1095.0,"1387100, 56717, 55396, 29096, 279173, 5992818, 2207484, 139791, 3080852, 56152"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
duration,68784.0,2.864139e+02,2.872576e+02,31.080000,2.584409e+04
width,68784.0,6.249342e+02,4.631691e+02,176.000000,1.920000e+03
height,68784.0,4.125722e+02,2.406155e+02,144.000000,1.080000e+03
bitrate,68784.0,6.937015e+05,1.095628e+06,8384.000000,7.628466e+06
framerate,68784.0,2.324132e+01,7.224848e+00,5.705752,4.800000e+01
i,68784.0,1.008683e+02,8.476479e+01,7.000000,5.170000e+03
p,68784.0,6.531692e+03,6.075872e+03,175.000000,3.049590e+05
b,68784.0,9.147854e+00,9.251618e+01,0.000000,9.407000e+03
frames,68784.0,6.641708e+03,6.153342e+03,192.000000,3.101290e+05
i_size,68784.0,2.838987e+06,4.325137e+06,11648.000000,9.082855e+07


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                           
codec    1            h264  31545  45.86
         2             vp8  18387  26.73
         3           mpeg4  12012  17.46
         4             flv   6840   9.94
o_codec  1           mpeg4  17291  25.14
         2             vp8  17277  25.12
         3             flv  17135  24.91
         4            h264  17081  24.83
video_id 1     3o_2-igmsyo    841   1.22
         2     3mWp-QQn-1M    841   1.22
         3     3fpg-LlHcks    841   1.22
         4     3fmy-4uvR9U    841   1.22
         5     3dI-kJGuWbY    841   1.22

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,9.03,0.214,NaN,1.415,0.415,log1p,196410.4,979737.9,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using label-per-sample grouped splits.


Repeat 0, Fold 0:
            Train N: 44897, Test N: 23887
            Target Distribution:
            	Train target distribution: 1.5827673967898477
            	Test target distribution: 1.5350661071911762
            Group Distribution video_id:
            	Train: 732
            	Test: 367
            
Repeat 0, Fold 1:
            Train N: 44898, Test N: 23886
            Target Distribution:
            	Train target distribution: 1.5655186446195792
            	Test target distribution: 1.5674862182876796
            Group Distribution video_id:
            	Train: 733
            	Test: 366
            
Repeat 0, Fold 2:
            Train N: 47773, Test N: 21011
            Target Distribution:
            	Train target distribution: 1.5512758234252642
            	Test target distribution: 1.6001395461260417
            Group Distribution video_id:
            	Train: 733
            	Test: 366
            
Repeat 1, Fold 0:
            Train N: 42377, Test N: 26407
       

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to video_transcoding_time_prediction/019d7392-553f-7bb2-b0f4-0d80ea446f63


019d7392-553f-7bb2-b0f4-0d80ea446f63
9517bb9ce6729db49a5a6b06d8c4eb95791f79efa995e3fbfaa9a94b789b5a5d
